In [ ]:
import sys; sys.path.append('..');
import MeshFEM
import mesh, elastic_sheet, energy, benchmark
import triangulation
from tri_mesh_viewer import TriMeshViewer
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import visualization

In [ ]:
import parametric_pillows

In [ ]:
alpha = 40

In [ ]:
from parametric_pillows import *

In [ ]:
# alpha: angle between spiral tangent and axis vector dtheta (not dr)
def logSpiralPlot(alpha = 70.0, radius = 1.0, minDist = 0.05, margin = 0.05, edgeLength = 0.02):
    alpha_rad = np.deg2rad(alpha)
    b = np.tan(alpha_rad)

    # Note: the logarithmic spiral is self-similar, so its scale is irrelevant.
    # We therefore use the unit-scaled logarithmic spiral, given in polar coordinates by:
    #   r = e^(b theta)
    # We evaluate the spiral at evenly spaced points along its arclength.
    sqrtTerm = np.sqrt(1 + 1 / (b * b))
    rForTheta      = lambda th: np.exp(b * th)
    thetaForR      = lambda r: np.log(r) / b
    thetaForArclen = lambda s: (1.0 / b) * (np.log(s + sqrtTerm) - np.log((sqrtTerm)))
    arclenForTheta = lambda th: sqrtTerm * (np.exp(b * th) - 1.0)

    def thetasForRadiusInterval(rmin, rmax):
        smin, smax = map(lambda r: arclenForTheta(thetaForR(r)), [rmin, rmax])
        nsubdiv = int(np.round((smax - smin) / edgeLength))
        return thetaForArclen(np.linspace(smin, smax, nsubdiv))

    pts, edges = circle(int(np.round(2 * radius * np.pi / edgeLength)))

    pts = []
    edges = []
    def generate_points(rs, thetas, rotation = 0):
        return np.column_stack((rs * np.cos(thetas + rotation), rs * np.sin(thetas + rotation)))

    numSectors = 2
    # Draw walls (spiral arms) dividing the circle into numSectors sectors
    # for numSectors in 2, 4, 8, ...
    # while True:
        # We approximate the channel thickness by multiplying the channel's
        # sector angle by the normal velocity of spiral arm (wall) points as
        # the arms are rotated at unit angular velocity.
        #       thickness ~= (2 * pi / numSectors) * r * sin(alpha)
        # where sin(alpha) is the (constant) angle between the curve's normal
        # and the radial axis vector dr.
        # Then we can solve for the minimum radius such that the thickness is >= minDist:
        #       (2 * pi / numSectors) * r * sin(alpha) >= minDist   ==>
        #       r >= (minDist * numSectors) / (2 * pi * sin(alpha))
    rmin = (minDist * numSectors) / (2 * np.pi * np.sin(alpha_rad))
        # The following version reproduces the original Matlab behavior (but
        # leads to tightly spaced channels for small alpha)
    rmin = max(minDist / 2, rmin) if numSectors > 2 else minDist / 2
    rmin = max(minDist / 2, rmin)
    rmax = radius - margin
    # if (rmin > rmax - edgeLength): break # admissible channel walls have shrunk below the target edge length
    thetas = thetasForRadiusInterval(rmin, rmax)
    rvalues = rForTheta(thetas)

    for arm in range(numSectors):
        # if ((numSectors > 1) and (arm % 2 == 0)): continue # even arms have already been drawn by previous passes
        newPts = list(generate_points(rvalues - arm * 0.05, thetas - arm * 0.1, arm * (30 / 180 * np.pi)))
        if (len(newPts) >= 2): # At least two points must be added to form a segment
            ptOffset = len(pts)
            pts += newPts
            print("end num pts ", len(pts)) 
            for i in range(len(newPts) - 1):
                edges.append((ptOffset + i, ptOffset + i + 1))
        # break
    # numSectors *= 2

    return pts, edges

In [ ]:
alpha = 20

In [ ]:
edgeLength = 0.02

In [ ]:
pts, edges = logSpiralPlot(alpha=alpha, edgeLength=edgeLength, minDist=0.2, margin=.0)

In [ ]:
edges += [[0, 132], [131, 263]]

In [ ]:
visualization.plot_line_segments(pts, edges)

In [ ]:
area = edgeLength ** 2 / 2

In [ ]:
V, F, edgeMarkers = triangulation.triangulate(pts, edges, triArea=area, outputPointMarkers=False, outputEdgeMarkers=True)


In [ ]:
old_m = mesh.Mesh(V, F)

In [ ]:
boundary = old_m.boundaryVertices()[tuple(old_m.boundaryLoops())]

In [ ]:
len(boundary)

In [ ]:
new_pts = old_m.vertices()[boundary]

In [ ]:
len(boundary)

In [ ]:
new_edges = [[boundary[i], boundary[i + 1]] for i in np.arange(-131, 0)]

In [ ]:
# new_edges

In [ ]:
visualization.plot_line_segments(old_m.vertices(), new_edges)

In [ ]:
vxs = old_m.vertices()

In [ ]:
boundary_vertices = list(boundary[-131:]) + [boundary[0]]


In [ ]:
# boundary_vertices


In [ ]:
second_vertices = []
second_faces = []

boundary_edge_map = {}

for i, vx in enumerate(old_m.vertices()):
    if i in boundary_vertices:
        boundary_edge_map[i] = i
    else:
        boundary_edge_map[i] = len(second_vertices) + len(vxs)
        second_vertices.append(vxs[i] + np.array([0, 0, 0.01]))

for i, face in enumerate(old_m.elements()):
    second_faces.append([boundary_edge_map[face[1]], boundary_edge_map[face[0]], boundary_edge_map[face[2]]])

In [ ]:
final_vertices = list(vxs) + list(second_vertices)

In [ ]:
final_faces = list(old_m.elements()) + list(second_faces)

In [ ]:
m = mesh.Mesh(final_vertices, final_faces)

In [ ]:
len(final_vertices)

In [ ]:
len(old_m.vertices()) * 2

In [ ]:
# visualization.plot_2d_mesh(m, width=12, height=12, )

In [ ]:
creases = np.array(new_edges[:30])

In [ ]:
# creases

In [ ]:
psi = energy.NeoHookeanYoungPoisson(2, 1e6, 0.3)
es = elastic_sheet.ElasticSheet(m, psi, creases)
es.thickness = 0.2
# pinVars, pinVerts = es.prepareRigidMotionPins()
creaseVars = np.arange(es.numCreases()) + es.creaseAngleOffset()

In [ ]:
esview = TriMeshViewer(es, wireframe=True, width=1024, height=768)
esview.materialLibrary.material(False).color='#537D8D'
esview.show()

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.niter = 1000

### Rerun the following cell to fold:

In [ ]:
fixedVars, hessianShift = [], 1e-6

In [ ]:
esview.update()

In [ ]:

    es.setCreaseAngles(es.getCreaseAngles()[0] + 0.5 * np.pi / 16 * np.ones(es.numCreases()))
    def iter_cb(prob, it):
        return
        if (it % 5 == 1):
            esview.update()
    #benchmark.reset()
    es.computeEquilibrium(loads=[], fixedVars=fixedVars + list(creaseVars), cb=iter_cb, opts=opts, hessianShift = hessianShift)
    esview.update()
    #benchmark.report()